# 6. A data quality report that counts NULLs correctly

**Track:** sql | **Difficulty:** easy | **Tags:** data quality, nulls, not in, duplicates

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("006-data-quality-report")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "shipments": """
UEFSMRUEFYAZFd4MTBWQAxUAEgAAwAwIDesABQEE6OoFBwQACQ0QALcNEAD8DQgA2A0IAO4NCAC5DQgAEg0wABgNCADBDRgAKA0Q
AMMNEAD4DQgAsg0IAKkNCACFDQgAzQ0IANYNCADTDQgAJw1IAHQNEADjDQgAHQ0YAMgNEACQDQgAGw0YAAcNCACmDRgAIA0QAPIN
EAC6DQgA3Q0IALsNCACwDQgACw0wALgNEACSDQgADw0YAIkNEADCDQgAZQ0IANsNCACMDQgAvw0IAPYNCACtDQgAiw0IAJMNCADP
DQgAdQ0IABANaADKDRAApQ0IAAANGAAcDQgA7w0YAHYNCAB7DQgAqg0IAA4NKACgDRAAfA0IANANCADzDQgACg0oANwNEAABDRAA
5g0QANENCAD5DQgAsQ0IAOcNCACrDQgAIQ04ANQNEADODQgAcw0IAB4NIADZDRAAaw0IAB8NGADVDRAA8Q0IAMcNCAD1DQgAlA0I
AJ4NCABoDQgAZA0IALMNCACbDQgAhw0IAKMNCAAZDWgAvg0QAPQNCACvDQgA4g0IAOQNCAARDTAAbg0QANoNCACiDQgAjw0IABcN
KACaDRAAAg0QACQNCADADRgAbw0IAKQNCADtDQgA/w0IAOoNCADhDQgA4A0IAHcNCABjDQgAig0IAKwNCAB/DQgAxQ0IAPsNCACO
DQgAYg0IAOUNCABnDQgAfQ0IAI0NCAChDQgAgg0IAOkNCAD6DQgAYQ0IAIYNCAC1DQgAnA0IAAUN8ACWDRAAcg0IAJUNCACBDQgA
nw0IALYNCABtDQgAZg0IAJcNCAB4DQgAFg1YAIANEACRDQgApw0IAMYNCABwDQgAvA0IAOwNCAAmDUAAvQ0QABoNEAAEDQgA0g0Y
AJgNCAAVDRgAaQ0QAKgNCAAjDRgA/g0QAN4NCAB6DQgACA0gAMsNEACIDQgA9w0IAMwNCADrDQgAEw0wAAwNCADEDRgAgw0IAJ0N
CAB5DQgAIg0oAN8NEAAlDRAAag0QAGwNCACEDQgAfg0IAMkNCACuDQgAFA04ANcNEAADDRAAcQ0QAAYNEACZDRAA/Q0IPLTqAAAA
AAAA8OoAAAAAAAAVABWyAxW6AywVnAMVEBUGFQYcGAgo6wAAAAAAABgIYeoAAAAAAAAWACgIKOsAAAAAAAAYCGHqAAAAAAAAEREA
AADZAfDYAwAAAJwDAQg1AAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7
Gzw9Pj9AQUJDREVGR0hJSktMTU5PUFFSU1RVVldYWVpbXF1eX2BhYmNkZWZnaGlqa2xtbm9wcXJzdHV2d3h5ent8fX5/gIGCg4SF
hoeIiYqLjI2Oj5CRkpOUlZaXmJmam5yWnZ6foKGio6SlpqeoqaqrrK2ur7CxLFKys3K0tba3uLm6u7y9G76/wMHCw8TFxscAABUE
FZAOFagHTBXiARUAEgAAiAcIq5wABQEAeA0IAF4NCAB3DQgASw0IANoNCAS7nAUvBAB9DQgAkg0IAKINCACzDQgAsQ0IAIcNCABz
DQgAkQ0IAEoNCADEDQgARw0IANQNCABhDQgEgZwJcADDDQgAvg0IAMgNCABCDQgA6g0IAFoNCAC9DQgA0A0IAI0NCAB/DQgAsA0I
AFYNCADWDQgAZw0IAKgNCACADQgA4w0IAOcNCAC3DQgAfA0IAG4NCABUDQgA4Q0IAMwNCACcDQgA4g0IAKANCACPDQgAiQ0IAFUN
CADdDQgAiw0IAJMNCAC1DQgAmg0IAGUNCACIDQgAsg0IAIINCABNDQgArg0IAMkNCADfDQgAQQ0IAKENCACpDQgAjg0IAHENCAB+
DQgARA0IAKYNCACKDQgAjA0IAJQNCACdDQgAQw0IAMoNCADpDQgAaA0IAFANCACZDQgA2w0IAGINCADZDQgA0g0IANMNCADODQgA
ZA0IAHQNCACEDQgA1Q0IAFMNCACXDQgA3A0IAGoNCACFDQgAew0IAFwNCABIDQgAqg0IALwNCABpDQgAuA0IAOANCABMDQgARg0I
ANENCACbDQgArA0IAJUNCDxXnAAAAAAAAE6cAAAAAAAAFQAVnAMVpAMsFZwDFRAVBhUGHBgI6pwAAAAAAAAYCEGcAAAAAAAAFg4o
COqcAAAAAAAAGAhBnAAAAAAAABERAAAAzgHwzRkAAAA6AQP+JAED/nQBBf73GAED/iABA/4uAQMCBzOAgGBAKBgOiIRiwWg4HpCI
ZEKpWC6YjGbD6Xg+gI8HFBKNSJiS6YRKqSqJFWvTKrkSL9gmVirIZjRTzXbDNRw53Y7Xk/kSIcEvBDzEgibhkPh4nDIVoad4MnJs
Rw8yqVPCFqYlY8AMNJ3PEZQVlcIOU2qVYR1diQ4sIavdcrtHlXf7BYA9Fm9YHIaMubYluQzOrcxnFDptqpwECPVSq6Uk1mG2a9Pe
UN2r9wDeDIMAFQQVngEViAFMFRQVABIAAE8cBAAAAHBucTEBCAhERUwFCAhIWUQFCAhQTlEFCAxCTFIyASAYQkxSMQMAAAEoAQ8I
YmxyBR8sTVVNOQQAAABCTFIzFQAVgAIVhAIsFZwDFRAVBhUGHDYIKARwbnExGARCTFIxEREAAACAAUASAAAATAED/lgBA/4cAQP+
TgUE8GkENRAhMUNBNRETNREVYRQxERIiExREFSIkMVRRFXNSNRgUIzRUQzNEJBJUVRMhQzIkMxMkEUNDMTIUJTQVEkU1QVUURCVC
QiUkIlMUIZVTEzFTM1IjQRRDMlVSIjUSNFQzMjFFUSVRAAAAFQQV4AEV5gFMFTgVABIAAHDwb6BOAACwTgAAqU4AAKZOAACnTgAA
mU4AAKxOAACzTgAAqE4AAK1OAACxTgAApE4AAJtOAACYTgAAn04AAKNOAACvTgAAnE4AAJ5OAACqTgAApU4AAKJOAACrTgAAnU4A
AKFOAACyTgAAmk4AAK5OAAAVABWWAhWeAiwVnAMVEBUGFQYcGASzTgAAGASYTgAAFgAoBLNOAAAYBJhOAAAREQAAAIsB8IoDAAAA
nAMBBTUgiEFCMecgg0xSZbHmnHsBspjMkiJMquo0ZLBzbLtHlifUxYuiHNEJGRtKhHDKpTSjUPNYG+jVEDY81A5KKqmr0GOBWKc1
cYZcDyoMiDgIfIzpkqYy5CqRzBQV77oFvRjSsveFzbCAR26yHWQI3XmrdSx2mVE8ks5zrgjKFjNMixUAFQQViAIVkAJMFUIVABIA
AIQB8IOmTgAAsU4AAKpOAACrTgAAtU4AAJ9OAACsTgAAs04AALdOAACuTgAAqU4AALJOAAC0TgAAmU4AAJ1OAACcTgAAoE4AAKRO
AACnTgAAnk4AAKhOAAC2TgAArU4AAJpOAAChTgAAsE4AALhOAACvTgAAmE4AAKVOAACXTgAAo04AAKJOAAAVABXiAhXqAiwVnAMV
EBUGFQYcGAS4TgAAGASXTgAAFhwoBLhOAAAYBJdOAAAREQAAALEB8LAbAAAAQgEDrioBBf7+KgED/hgBBf7vPAEN/v77ffsBBjFA
IAwDURgHYiTKEjCN4jQPFEXHARVHIk2GQFWWZTgHM1zTAV2KEEAYRSlMQWUatilKglXQZlmOtRhHkjHDxAWbAwzMwmQQ4UjCowjd
0xmB9VjMAVTGEwnYIDCSg3lZUgVUEShBkiRSdHzZBnxYRj3B8QVTMmWSdkXOkVhUchgVxWTf8XTRYDGCIBTOYhzCAB4VBBXwFxWW
D0wV/gIVABIAAPgLIHE9CtejsEJAAA0B2FK4HoXr8UJArkfhehRuPUD2KFyPwnVAQEjhehSuRyJAhetRuB5FPEBmZmZmZiY5QM3M
zMzMzD0NGAQ3QAFPEHA9CkAUBUEgE0CamZmZmdk8CQgEGSIJSAzHOUB7ASEM+i9A1wEuDNc6QDMFAQw5QMP1AXEEP0ABhyxRuDZA
KVyPwvUoKEAFqARwGQlABEcbCVAImfU/BSAECEINiABDCSAEhzgNSAAlDQggH0CkcD0K1+M0CQgEIzYJ2AS1NAkIADUNGASjNQlA
BAdAKQAErj4J0AT9OwlwCFk8QAWPBLxADVgAQwnACFc8QCUfDEFDQAoF0QBBCRAEIT4NyAQ/QCVYEFEyQB+FIUEAJg1QADsJSARi
QwkIBMInCZAE9RgJ0CTsQ0DsUbgehas1DbgAIgkIBMNAKUAEmkApiASmMw14BO8/IVcIevRCCcgERzwJQAjrE8AFSAQrQQlYBMwo
LaAEOkAhXghcjwQJmAReMSnIBMU9DdgAQwnIBMFDCRgEhRsJOASULgnABBkwKSgEbkEJcASHOklIBEAyCTgE4SIJcARMQykABBE/
CbAAJjEgESgAEy3oACEJsAQrNgkoBOYwKUAAOA1oBK5DKZAEPRAJYARhNymQBFwHDRAANglwBMwZCdgEDy8JEARMNgkgBKE6CQgE
4S4NCAAlCegEd0ApWAijG8ApwAAxCTgEDEEpWARKQAlQBI9DCcgAUREwACIJkAT9QQ3AAB4pIARUQinIBHUoSdgE8z8NGAA7CUgE
TzQpcAQeFy34ACpJ4ATwQgkoBBQGDSAANAkwBI8uCUgEszcJEATvQw1wAEANMAAECdAEYTAtqAAaDRgAFCkgBJxDKagErjENOAAi
CXgEAjMJeACwEdAAQgloBDMtCSgEqCAp6ATZPQ1AAAsNWAAwLTAAMwlIBEI6KagEeDkNuAAPCTAA7lFgABIpUARMPy1QAEMJCAQK
BQ1IAEEJqATcIA2AACEJ6ARPOi2gAEEJEAQPQCmABNEhDXAAMQlwBF46DYgAIAnQBHAyCUgE8zxJSABmjQAEoyVJqARHIwmQCMz8
P2WYBNEjDegAQAlQBB42KUgEBT1J0AAALbgEjz8JgCBR9D+4HoXrUTiNoAToM0m4AEttCARHCwk4BEA2KQgAzg0YBIc+CSgE60AN
qAT1P2mABBnAZbgEtEMJgATeOgkIBJ4iDcgEPUBlwATrKwkYAH5NmAQKEQloBIAkCegE5i8tuAAlKagEVDQJsASRNwkYKCg7QKRw
PQrXoxNAFQAVvgMVxgMsFZwDFRAVBhUGHBgIj8L1KFzvQ0AYCKRwPQrXoxvAFgwoCI/C9Shc70NAGAikcD0K16MbwBERAAAA3wHw
3hEAAAAD1xABA96mAQED/pwBAQX+HwgzAAECAwQFBgcICQoLDA0ODxAREhMUFRYXARgZGhscHR4fICEiIyQlJicoKQEqKywtLi8w
MTIzNDUXNjc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWprAWxtbm9wcXJzdHV2d3h5
ent8fX5/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUjpWWl5iZmpucnZ6foKGio6SlpqeoqSdMqqtrrK2ur7CxsrO0F7W2t7i5uru8vb4V
BBl8NQAYBnNjaGVtYRUMABUEJQIYC3NoaXBtZW50X2lkABUEJQIYCG9yZGVyX2lkABUMJQIYDndhcmVob3VzZV9jb2RlJQBMHAAA
ABUCJQIYCnNoaXBwZWRfb24lDExsAAAAFQIlAhgMZGVsaXZlcmVkX29uJQxMbAAAABUKJQIYCXdlaWdodF9rZwAWnAMZHBlsJgAc
FQQZNQAGEBkYC3NoaXBtZW50X2lkFQIWnAMW2B0WvhEmiA0mCBwYCCjrAAAAAAAAGAhh6gAAAAAAABYAKAgo6wAAAAAAABgIYeoA
AAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAnAMAAAAmABwVBBk1AAYQGRgIb3JkZXJfaWQVAhacAxbSEhbyCyaQGSbGERwY
COqcAAAAAAAAGAhBnAAAAAAAABYOKAjqnAAAAAAAABgIQZwAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYOjgMAAAAmABwV
DBk1AAYQGRgOd2FyZWhvdXNlX2NvZGUVAhacAxaKBBb4AybgHia4HRw2CCgEcG5xMRgEQkxSMRERABksFQQVABUCABUAFRAVAgA8
Fs4MGQYZJgiUAwAAACYAHBUCGTUABhAZGApzaGlwcGVkX29uFQIWnAMW+gQWiAUmtiMmsCEcGASzTgAAGASYTgAAFgAoBLNOAAAY
BJhOAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAnAMAAAAmABwVAhk1AAYQGRgMZGVsaXZlcmVkX29uFQIWnAMW7gUW/gUm6Cgm
uCYcGAS4TgAAGASXTgAAFhwoBLhOAAAYBJdOAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYcgAMAAAAmABwVChk1AAYQGRgJd2Vp
Z2h0X2tnFQIWnAMW1BwWghQm7jsmtiwcGAiPwvUoXO9DQBgIpHA9CtejG8AWDCgIj8L1KFzvQ0AYCKRwPQrXoxvAEREAGSwVBBUA
FQIAFQAVEBUCADwpBhkmDJADAAAAFvBbFpwDJggWsEAAGRwYDEFSUk9XOnNjaGVtYRisBC8vLy8vNWdCQUFBUUFBQUFBQUFLQUF3
QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFZQUFBQXNBUUFBNUFBQUFLd0FBQUJ3QUFBQU9B
QUFBQVFBQUFEOC92Ly9BQUFCQXhBQUFBQWNBQUFBQkFBQUFBQUFBQUFKQUFBQWQyVnBaMmgwWDJ0bkFBQUFtdi8vL3dBQUFnQXMv
Ly8vQUFBQkNCQUFBQUFnQUFBQUJBQUFBQUFBQUFBTUFBQUFaR1ZzYVhabGNtVmtYMjl1QUFBQUFNNy8vLzhBQUFBQVlQLy8vd0FB
QVFnUUFBQUFKQUFBQUFRQUFBQUFBQUFBQ2dBQUFITm9hWEJ3WldSZmIyNEFBQUFBQmdBSUFBWUFCZ0FBQUFBQUFBQ1kvLy8vQUFB
QkJSQUFBQUFrQUFBQUJBQUFBQUFBQUFBT0FBQUFkMkZ5WldodmRYTmxYMk52WkdVQUFBUUFCQUFFQUFBQXpQLy8vd0FBQVFJUUFB
QUFIQUFBQUFRQUFBQUFBQUFBQ0FBQUFHOXlaR1Z5WDJsa0FBQUFBTHovLy84QUFBQUJRQUFBQUJBQUZBQUlBQVlBQndBTUFBQUFF
QUFRQUFBQUFBQUJBaEFBQUFBa0FBQUFCQUFBQUFBQUFBQUxBQUFBYzJocGNHMWxiblJmYVdRQUNBQU1BQWdBQndBSUFBQUFBQUFB
QVVBQUFBQT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRlsHAAAHAAAHAAAHAAAHAAAHAAAAIcFAABQQVIx
""",
    "warehouses": """
UEFSMRUEFVAVRkwVChUAEgAAKBwEAAAAUE5RMQEICEJMUhEIADIBECxERUwxBAAAAEhZRDEVABUWFRosFQwVEBUGFQYcNgIoBFBO
UTEYBEJMUjEREQAAAAsoAgAAAAMfAwOIRgAVBBV0FXZMFQoVABIAADp0BAAAAFB1bmUJAAAAQmVuZ2FsdXJ1BQAAAERlbGhpARZc
SHlkZXJhYmFkCwAAAENsb3NlZCBzaXRlFQAVFhUaLBUMFRAVBhUGHDYAKARQdW5lGAlCZW5nYWx1cnUREQAAAAsoAgAAAAwBAwNI
NAIVBBk8NQAYBnNjaGVtYRUEABUMJQIYDndhcmVob3VzZV9jb2RlJQBMHAAAABUMJQIYBGNpdHklAEwcAAAAFgwZHBksJgAcFQwZ
NQAGEBkYDndhcmVob3VzZV9jb2RlFQIWDBbIARbCASZqJggcNgIoBFBOUTEYBEJMUjEREQAZLBUEFQAVAgAVABUQFQIAPBYoGQYZ
JgIKAAAAJgAcFQwZNQAGEBkYBGNpdHkVAhYMFvYBFvwBJtwCJsoBHDYAKARQdW5lGAlCZW5nYWx1cnUREQAZLBUEFQAVAgAVABUQ
FQIAPBZeGQYZJgAMAAAAFr4DFgwmCBa+AwAZHBgMQVJST1c6c2NoZW1hGOwBLy8vLy82Z0FBQUFRQUFBQUFBQUtBQXdBQmdBRkFB
Z0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQUlBQUFCQUFBQUFCQUFBQU5qLy8vOEFBQUVGRUFBQUFCZ0FB
QUFFQUFBQUFBQUFBQVFBQUFCamFYUjVBQUFBQU1ELy8vOFFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFVUUFBQUFKQUFB
QUFRQUFBQUFBQUFBRGdBQUFIZGhjbVZvYjNWelpWOWpiMlJsQUFBRUFBUUFCQUFBQUFBQUFBQT0AGCBwYXJxdWV0LWNwcC1hcnJv
dyB2ZXJzaW9uIDI1LjAuMRksHAAAHAAAACQCAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "006-data-quality-report"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

shipments = spark.read.parquet((_data_dir / "shipments.parquet").resolve().as_posix())
shipments.createOrReplaceTempView("shipments")
warehouses = spark.read.parquet((_data_dir / "warehouses.parquet").resolve().as_posix())
warehouses.createOrReplaceTempView("warehouses")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'failed_rows': 'bf8fbf5c6b0722f7f19fdb0c9e5bbd9c22784d7d0edc33e530e18345da6fc75d',
                         'rule_name': '98989e262cf2d9721efea046954936e5e32f4f6cc329c64eadfd4ec8f003c0ed'},
 'columns': [['failed_rows', 'bigint'], ['rule_name', 'string']],
 'fingerprint': '508eefa9a62a7d541a320aeca8aefca46b4bfa596341e39718910fcc82f2fb32',
 'order_matters': False,
 'row_count': 6}

check = make_check(EXPECTED)

## The situation

Before the `shipments` table is published each morning, a check has to report how
many rows break each data quality rule. The report goes to a dashboard, so it must
always list every rule, including the ones with no failures today.

## Input tables

### `shipments`

Supposed to be one row per shipment.

| Column | Type | Description |
|---|---|---|
| `shipment_id` | BIGINT | Supposed to be unique. |
| `order_id` | BIGINT | The order being shipped. |
| `warehouse_code` | STRING | The warehouse it shipped from. |
| `shipped_on` | DATE | Never NULL. |
| `delivered_on` | DATE | NULL while the shipment is in transit. |
| `weight_kg` | DOUBLE | Weight of the parcel. |

### `warehouses`

The list of warehouses.

| Column | Type | Description |
|---|---|---|
| `warehouse_code` | STRING | The code used in `shipments`. One closed site has a NULL code. |
| `city` | STRING | Where the warehouse is. |

## Requirement

Return one row for each of these six rules, with the number of rows in `shipments`
that fail it. A rule that no row fails has `failed_rows` = 0.

| `rule_name` | A row fails when |
|---|---|
| `missing_shipment_id` | `shipment_id` is NULL. |
| `missing_order_id` | `order_id` is NULL. |
| `duplicate_shipment_id` | Its `shipment_id` has already appeared. For an id that occurs three times, two rows fail. |
| `delivered_before_shipped` | `delivered_on` is earlier than `shipped_on`. A shipment still in transit does not fail. Delivery on the shipping date does not fail. |
| `invalid_weight` | `weight_kg` is NULL, zero or negative. |
| `unknown_warehouse` | `warehouse_code` is NULL, or does not exactly match a code in `warehouses`. |

## Expected output

Exactly six rows.

| Column | Type | Description |
|---|---|---|
| `rule_name` | STRING | One of the six names above. |
| `failed_rows` | BIGINT | Number of failing rows. |

In [ ]:
shipments.show(5)
warehouses.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `shipments`, `warehouses`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)